# LOGAN: Prodigal genes the model scores low

**Draft caption.** The second discordant class: complete Prodigal genes scoring below the 0.80-precision threshold. (a) Share below the threshold by gene length, from the scan's 1-in-1,000 sample of complete Prodigal genes (880,405). (b) Full-length structural hit rate for 10,269 low-scoring genes and 8,769 high-scoring genes (≥ the 0.90 threshold) matched to them by length (from 50 aa; high-scoring genes below 50 aa barely exist); bins with fewer than 100 genes are not drawn. (c) Full-length structural hit rate of every length-matched arm (five databases), including the frameshift-shadow control.

**Outputs:** `../images/figL10_prodigal_lowscore.pdf` (vector) and `../images/figL10_prodigal_lowscore.png` (300 dpi).

**Data** (plotting-ready, committed; produced by `../prep/logan_prepare_data.py` on Setonix):

- `../data/figL10_lowscore_population.tsv.gz` (9 rows; sha256 `3e42feb41e60…`): share of complete Prodigal genes below t0.80, by length. Source on Setonix (`$LOGAN_ROOT`): `qc/candidates_extra/lowscore/lowscore_population.tsv`.
- `../data/figL10_lowscore_evidence_by_length.tsv.gz` (8 rows; sha256 `162f36ea61fc…`): low- vs high-score Prodigal genes, length-matched. Source on Setonix (`$LOGAN_ROOT`): `qc/candidates_extra/lowscore/lowscore_evidence_by_length.tsv`.
- `../data/figL10_all_arms_evidence.tsv.gz` (8 rows; sha256 `85ec60c0bfb1…`): hit / full-length rates of every arm over all five databases. Source on Setonix (`$LOGAN_ROOT`): `qc/candidates_extra/lowscore/all_arms_evidence.tsv`.

**Upstream.** LOGAN v1.2 metagenomes (50,009), genome_entropy #102 (closed by PR #115); analysis scripts `Validation/logan/scripts/` at commit `06080b4`. Full run provenance: `../data/logan_provenance.json`.

**Run:** `jupyter nbconvert --to notebook --execute figL10_prodigal_lowscore.ipynb` from `manuscript/notebooks/`. Needs only pandas, numpy, matplotlib and seaborn; no HPC or remote paths.

In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

DATA, IMAGES = Path("../data"), Path("../images")
IMAGES.mkdir(exist_ok=True)
plt.rcParams["svg.hashsalt"] = "genome_entropy"   # deterministic SVG ids if SVG is ever written

# Colour slots (fixed order, colour-blind-checked categorical palette used across the LOGAN figures)
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
sns.set_theme(context="paper", style="ticks", font_scale=1.15, rc={
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.facecolor": SURFACE, "figure.facecolor": "white", "grid.color": GRID,
    "axes.grid": True, "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "lines.linewidth": 2, "font.family": "DejaVu Sans"})


def panel(ax, letter, x=-0.08, y=1.04):
    ax.text(x, y, letter, transform=ax.transAxes, fontsize=15, fontweight="bold", va="bottom", ha="right")


def save(fig, name):
    """Vector PDF (fixed metadata, so reruns are identical) and a 300-dpi PNG preview."""
    fig.savefig(IMAGES / f"{name}.pdf", bbox_inches="tight", metadata={"CreationDate": None, "ModDate": None})
    fig.savefig(IMAGES / f"{name}.png", dpi=300, bbox_inches="tight", metadata={"Software": None})
    plt.close(fig)
    print("wrote", IMAGES / f"{name}.pdf", IMAGES / f"{name}.png")

In [ ]:
P = pd.read_csv(DATA / "figL10_lowscore_population.tsv.gz", sep="\t", index_col=0).drop(index="all")
L = pd.read_csv(DATA / "figL10_lowscore_evidence_by_length.tsv.gz", sep="\t", index_col=0)
A = pd.read_csv(DATA / "figL10_all_arms_evidence.tsv.gz", sep="\t", index_col=0)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), gridspec_kw={"width_ratios": [1, 1.1, 1]})
ax = axes[0]
ax.bar(range(len(P)), P.below_t80, color=SLOTS[1], width=0.7)
ax.set_xticks(range(len(P)), P.index, rotation=30)
ax.set_ylim(0, 1.08)
ax.set_xlabel("gene length (aa)")
ax.set_ylabel("complete Prodigal genes below threshold")
for i, v in enumerate(P.below_t80):
    ax.text(i, v + 0.015, f"{100 * v:.1f}%" if v < 0.1 else f"{100 * v:.0f}%", ha="center", fontsize=8.5, color=INK2)
panel(ax, "a")
ax = axes[1]
x = np.arange(len(L))
for arm, c, lab in (("prodigal_hi", SLOTS[0], "score ≥ 0.90 threshold"), ("prodigal_lo", SLOTS[1], "score < 0.80 threshold")):
    ok = L[f"queries_{arm}"].fillna(0).to_numpy() >= 100
    ax.plot(x[ok], L[f"full_struct_{arm}"].to_numpy()[ok], marker="o", ms=8, color=c, label=f"Prodigal genes, {lab}")
ax.set_xticks(x, L.index, rotation=30)
ax.set_ylim(0, 1)
ax.set_xlabel("gene length (aa)")
ax.set_ylabel("full-length structural hit rate")
ax.legend(loc="center right", fontsize=9)
panel(ax, "b")
ax = axes[2]
arms = ["prodigal", "prodigal_hi", "candidate", "prodigal_lo", "frameshift", "antisense", "intergenic_lo",
        "candidate_shuffled_3di"]
names = {"prodigal": "complete Prodigal genes", "prodigal_hi": "Prodigal genes, high score",
         "candidate": "candidates", "prodigal_lo": "Prodigal genes, low score", "frameshift": "frameshift shadows",
         "antisense": "antisense shadows", "intergenic_lo": "low-score intergenic",
         "candidate_shuffled_3di": "candidates, shuffled 3Di"}
cols = {"prodigal": SLOTS[0], "prodigal_hi": SLOTS[0], "candidate": SLOTS[1], "prodigal_lo": SLOTS[1],
        "frameshift": SLOTS[4], "antisense": SLOTS[3], "intergenic_lo": SLOTS[2], "candidate_shuffled_3di": SLOTS[6]}
v = A.loc[arms, "fullcov_rate_struct"]
ax.barh(range(len(arms)), v, color=[cols[a] for a in arms], height=0.6)
for i, xv in enumerate(v):
    ax.text(xv + 0.01, i, f"{100 * xv:.3f}%" if xv < 0.001 else f"{100 * xv:.2f}%" if xv < 0.01 else f"{100 * xv:.1f}%",
            va="center", fontsize=9, color=INK2)
ax.set_yticks(range(len(arms)), [names[a] for a in arms])
ax.invert_yaxis()
ax.set_xlim(0, 1.05)
ax.set_xlabel("full-length structural hit rate")
panel(ax, "c", x=-0.55)
fig.tight_layout()
save(fig, "figL10_prodigal_lowscore")